# NCM GPT-1 在 Kaggle 訓練（T4 × 2）

設定（右上角）：
- Accelerator: **GPU T4 x2**
- Internet: **On**
- 用 **Save & Run All**（不要用互動模式長跑，閒置約 20 分鐘會被中斷）

每一場最多 12 小時。不要重裝 PyTorch。

下一場請掛上 **COMPLETE 的 kernel 第 5 版 Output**（generator step 27595，tokenizer 已在 100000 步完成）。`tokenizer/status.json` 為 `finished: true` 時會跳過 tokenizer，generator 從該檢查點續訓。不要重跑 tokenizer。

T4（sm_75）維持梯度檢查點、chunked block attention，以及 Ampere 以前的 fp16 + GradScaler（不要用模擬 bf16）。有檢查點之後，預設每卡 batch **4**。kernel v4 在**沒有**檢查點時 batch 4 會 OOM；batch 8 先不要當預設，確認 4 放得下再試。`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`。

相對第 5 版、從 step 27595 接上時會一起生效的設定：
- `data.max_windows` 從 65536 提到 **524288**（約 2.68 億 token，不再反覆同一段約 3350 萬 token）。打包結果寫入 `artifacts/packed_windows.pt`，同一上限的下一場可重用。
- `alpha: 0.5` 會把損失從論文 ELBO（`alpha: 0`、每個碼位置等權）改成較重的中／粗層。這是刻意實驗，不是論文設定。loss 不要和 v5 的 alpha 0 曲線直接比較。
- 每 1000 步在 rank 0 對 256 個保留視窗算 val loss 與各層 CE。jsonl 保留舊欄位，並多 grad norm、data epoch、corrupt_p、全部層的 CE。

若這份程式還沒在 `main` 上，請把含這些改動的倉庫當成 Input dataset 掛上（下面的 cell 會優先複製裡面的 `train_tokenizer.py`），不要讓 clone 到的舊 `main` 蓋掉它。

In [ ]:
# 只補訓練依賴，不要重裝 torch
import os, subprocess, sys
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "omegaconf", "hydra-core", "datasets", "regex", "tqdm"])
print("deps ok")

In [ ]:
import os, sys, subprocess
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

REPO = "https://github.com/nicholashui/ncm-gpt1.git"
ROOT = Path("/kaggle/working/ncm-gpt1")
if not (ROOT / "train_tokenizer.py").exists():
    # 若已把程式當成 Input dataset 掛進來，直接複製；否則 clone。
    # 在這份改動進入 main 之前，請掛含 batch 4 / alpha 0.5 / max_windows 524288 的程式，避免 clone 到舊 main。
    local = None
    for cand in Path("/kaggle/input").glob("*/train_tokenizer.py") if Path("/kaggle/input").exists() else []:
        local = cand.parent
        break
    if local:
        import shutil
        shutil.copytree(local, ROOT, dirs_exist_ok=True)
        print("copied repo from", local)
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(ROOT)])
        print("cloned", REPO)
else:
    print("repo already present")

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("cwd", Path.cwd())

In [ ]:
import torch
print("cuda:", torch.cuda.is_available(), "gpus:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        cap = torch.cuda.get_device_capability(i)
        print(i, torch.cuda.get_device_name(i), "mem", round(torch.cuda.get_device_properties(i).total_memory/1024**3, 1), "GiB", "cap", cap)
assert torch.cuda.is_available(), "請把 Accelerator 設成 GPU T4 x2"
assert torch.cuda.device_count() >= 1

訓練語料預設是開源 BookCorpus 重建版（與 2018 GPT-1 的書籍語料同一類）。若下載失敗，腳本會改用 MiniPile，再不行才用 TinyStories。檢查點寫到 `/kaggle/working/artifacts`。

掛上第 5 版完整 Output 後，tokenizer 會被跳過，generator 從 step 27595 的 `latest.pt` 載入權重與 optimizer。視窗池與 `alpha` 用這一版設定（較大的 `max_windows`、`alpha: 0.5`）。規則切分器沒有權重，仍會按句打包，並寫下 `artifacts/splitter/status.json`。

In [ ]:
import os, subprocess, sys
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

out = Path("/kaggle/working/artifacts")
out.mkdir(parents=True, exist_ok=True)

# 若上一場的 Output 已掛成 Input，自動接續。
# 下一場應掛 COMPLETE kernel v5（generator step 27595 + 已完成的 tokenizer）。
cmd = [
    sys.executable, "scripts/train_kaggle.py",
    "--hours", "11.5",
    "--reserve_minutes", "15",
    "--output_dir", str(out),
    "--config", "config/kaggle.yaml",
    "--batch_size", "4",
    "--gradient_checkpointing",
]
print("running:", " ".join(cmd), flush=True)
rc = subprocess.call(cmd)
print("exit", rc)
if rc != 0:
    raise SystemExit(rc)

In [ ]:
from pathlib import Path

def show(stage):
    p = Path("/kaggle/working/artifacts") / stage / "status.json"
    print("=" , stage, "=")
    if p.exists():
        print(p.read_text())
    else:
        print("no status.json yet")
    log = Path("/kaggle/working/artifacts") / stage / f"{stage}_log.jsonl"
    if log.exists():
        lines = log.read_text(encoding="utf-8").strip().splitlines()
        print("log lines", len(lines))
        if lines:
            print("last:", lines[-1])

show("tokenizer")
show("generator")